# Chapter 09: Complex Numbers

Source orientation: printed pages 135-147; PDF pages 153-165. I inspected the rendered source span privately, especially the sections on rational and real numbers, the Argand diagram, modulus and amplitude, De Moivre's theorem, Euler's formula, roots of equations, and conformal transformations. This notebook is original teaching material: it does not copy textbook prose, exercise text, screenshots, page crops, or source figures.

## Chapter Goal

Complex numbers become geometry when every algebraic operation is also a motion, a scale, an angle rule, or a circle-preserving map. The chapter begins with the pressure that forces the number system to grow: rational numbers leave gaps, real numbers fill the line, and the equation `x^2 + 1 = 0` asks for a plane rather than another point on the same line. The Argand diagram is not decoration. It is the coordinate system in which addition, multiplication, conjugation, reciprocals, powers, roots, and fractional linear maps can all be inspected.

The notebook follows that path as a sequence of visual claims. First, it places rationals, selected irrationals, and complex numbers into one computational ladder. Second, it draws the algebra of `(a + bi)(x + yi)` as a real 2-by-2 matrix so the learner can see rotation and dilation rather than memorize a formula. Third, it uses modulus and amplitude to show why powers of a unit complex number walk around a regular polygon, giving De Moivre's theorem a visible orbit. Fourth, it rebuilds Euler's formula from partial sums, so `exp(i theta)` can be recognized as a point moving on the unit circle. Fifth, it treats roots of equations as geometric constellations: a polynomial is not just solved, it distributes its roots in the plane with residuals that can be checked. Finally, it turns to conformal transformations, where translations, dilative rotations, conjugations, inversions, and linear fractional maps preserve circles and angles in ways that can be probed numerically.

The source span is short but dense, so the notebook is intentionally organized around inspection targets. Every artifact asks a precise question: what is being preserved, what changes under the operation, and which calculation certifies that the visual observation is structural rather than accidental?

## Visualization Storyboard

| Section | Representation | Artifact | Inspection target | Check |
|---|---|---|---|---|
| Number extensions | number line plus Argand plane | `number-system-argand-ladder.png` | why the real line is not enough for `i` | rational approximation table and source-span summary |
| Complex arithmetic | Argand parallelogram and multiplication matrix | `argand-addition-multiplication.png` | multiplication by `a + bi` rotates and scales every vector | matrix/product equality, modulus product, argument addition |
| Modulus and amplitude | unit roots and logarithmic spiral | `modulus-amplitude-demoivre-roots.png` | powers turn angle addition into regular polygons and spirals | `omega^n = 1`, constant angle step, radial ratio |
| Euler formula | partial sums in the complex plane | `euler-formula-series-convergence.png` | the exponential series converges to `cos theta + i sin theta` | numeric convergence and `exp(i*pi)+1` residual |
| Roots of equations | plotted polynomial roots | `polynomial-root-constellations.png` | roots are plane configurations with checkable residuals | CSV residual table, max residual bound |
| Conformal maps | grid and circle images under linear fractional maps | `circle-preserving-conformal-maps.png`, `linear-fractional-map-lab.html` | similarities, inversions, and homographies preserve circle/line families and local angles | finite-difference angle preservation, determinant nonzero |

Library routing is deliberately modest. Matplotlib is used for durable static diagrams where exact labels and side-by-side comparisons matter. Plotly is used for the conformal map lab because toggling maps and inspecting traces is more useful than a single frozen picture. SymPy and NumPy provide symbolic and numerical checks so the notebook can defend its claims. No shared generator is used; the cells below build the chapter-specific artifacts directly.

In [ ]:
from __future__ import annotations

from fractions import Fraction
from pathlib import Path
import cmath
import math
import sys

import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch
import numpy as np
import plotly.graph_objects as go
import sympy as sp
from IPython.display import Markdown, display

CHAPTER_NO = 9
HERE = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [HERE, *HERE.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not locate the Introduction-to-Geometry book root")
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import (
    assert_artifacts,
    display_artifact,
    ensure_chapter_artifact_dirs,
    write_csv,
    write_json,
)
from utils.course import chapter_by_no

CHAPTER = chapter_by_no(CHAPTER_NO)
DIRS = ensure_chapter_artifact_dirs(CHAPTER_NO, BOOK_ROOT)
FIG = DIRS["figures"]
HTML = DIRS["html"]
CHECKS = DIRS["checks"]
TABLES = DIRS["tables"]
DATA = DIRS["data"]

# Remove bootstrap-only artifacts so the audit sees the chapter-specific pass.
for stale in [
    FIG / "concept_configuration.svg",
    FIG / "parameter_experiment.svg",
    TABLES / "artifact_manifest.csv",
]:
    if stale.exists():
        stale.unlink()

COLORS = {
    "ink": "#111827",
    "muted": "#6b7280",
    "blue": "#2563eb",
    "green": "#059669",
    "orange": "#d97706",
    "red": "#dc2626",
    "purple": "#7c3aed",
}

visual_artifacts: list[Path] = []
check_artifacts: list[Path] = []
table_artifacts: list[Path] = []
computed_checks: dict[str, object] = {}

def save_png(fig: plt.Figure, filename: str) -> Path:
    path = FIG / filename
    fig.savefig(path, dpi=165, bbox_inches="tight")
    plt.close(fig)
    visual_artifacts.append(path)
    return path

def display_many(paths: list[Path], width: int = 760) -> None:
    for path in paths:
        display(Markdown(f"### {path.name}"))
        display_artifact(path, width=width)

def rel(path: Path) -> str:
    return path.resolve().relative_to(BOOK_ROOT.resolve()).as_posix()

def angle_diff(a: float, b: float) -> float:
    return float((a - b + np.pi) % (2 * np.pi) - np.pi)


## 1. From Number Line To Argand Plane

The chapter's opening sections are about pressure on the number system. Rational numbers let us count, compare, and divide, but a rational lattice cannot fill every geometric length. Real numbers complete the line for ordinary metric geometry, yet the equation `x^2 + 1 = 0` still has no point on that line. The complex plane solves that problem by changing the stage: instead of squeezing a new number into one dimension, it treats `x + iy` as the point `(x, y)`.

The first figure should be read left to right. The rational samples are intentionally thin because the issue is not that rationals are rare; they are dense. The issue is that density is not completeness. The selected real markers show familiar lengths that arise from geometry and analysis. The final panel turns the missing root of `x^2 + 1` into a vertical unit direction. Once that step is taken, every complex number has both a horizontal real part and a vertical imaginary part, and algebra can now draw pictures.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4.6))

ax = axs[0]
for q in [Fraction(n, d) for d in range(1, 8) for n in range(-2*d, 2*d + 1)]:
    x = float(q)
    if -2.1 <= x <= 2.1:
        ax.plot(x, 0, marker="|", color=COLORS["blue"], markersize=14, alpha=0.5)
ax.axhline(0, color=COLORS["ink"], lw=1.2)
ax.set_title("rational samples on a line")
ax.set_xlim(-2.1, 2.1)
ax.set_ylim(-0.5, 0.5)
ax.set_yticks([])
ax.set_xlabel("x")
ax.grid(axis="x", alpha=0.25)

ax = axs[1]
real_markers = [(-math.sqrt(2), "-sqrt(2)"), (-1, "-1"), (0, "0"), (1, "1"), (math.sqrt(2), "sqrt(2)"), (math.pi / 2, "pi/2")]
ax.axhline(0, color=COLORS["ink"], lw=1.2)
for x, label in real_markers:
    ax.scatter([x], [0], s=55, color=COLORS["orange"])
    ax.text(x, 0.12, label, ha="center", va="bottom", fontsize=9)
ax.set_title("real line: metric lengths live here")
ax.set_xlim(-2.1, 2.1)
ax.set_ylim(-0.45, 0.55)
ax.set_yticks([])
ax.set_xlabel("x")
ax.grid(axis="x", alpha=0.25)

ax = axs[2]
ax.axhline(0, color=COLORS["ink"], lw=1)
ax.axvline(0, color=COLORS["ink"], lw=1)
for z, label, color in [(1 + 0j, "1", COLORS["blue"]), (1j, "i", COLORS["red"]), (-1 + 0j, "-1", COLORS["blue"]), (-1j, "-i", COLORS["red"]), (1.2 + 0.8j, "1.2 + 0.8i", COLORS["green"])]:
    ax.scatter([z.real], [z.imag], s=70, color=color)
    ax.text(z.real + 0.06, z.imag + 0.06, label, fontsize=9)
ax.add_patch(Circle((0, 0), 1, fill=False, ls="--", lw=1.2, ec=COLORS["muted"]))
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-1.7, 1.7)
ax.set_ylim(-1.4, 1.4)
ax.set_title("Argand plane: x + iy becomes (x, y)")
ax.set_xlabel("real part")
ax.set_ylabel("imaginary part")
ax.grid(alpha=0.25)

fig.suptitle("The complex plane is the next geometric home, not another tick on the same line", fontsize=14)
number_ladder_path = save_png(fig, "number-system-argand-ladder.png")

approx_rows = []
for name, value in [("sqrt(2)", math.sqrt(2)), ("pi", math.pi), ("e", math.e)]:
    for denominator in [8, 32, 128]:
        frac = Fraction(value).limit_denominator(denominator)
        approx_rows.append({
            "quantity": name,
            "max_denominator": denominator,
            "rational": f"{frac.numerator}/{frac.denominator}",
            "absolute_error": abs(float(frac) - value),
        })
approx_table = write_csv(TABLES / "rational-approximation-ladder.csv", approx_rows)
table_artifacts.append(approx_table)
computed_checks["number_ladder"] = {
    "source_pdf_pages": CHAPTER["pdf"],
    "rational_approximation_rows": len(approx_rows),
    "i_squared": (1j * 1j).real,
}

display_many([number_ladder_path])


## 2. Complex Arithmetic As Plane Motion

Addition in the Argand diagram is vector addition: the parallelogram rule is literally the equation `(u + v) - u = v`. Multiplication is more surprising. If `a = p + qi`, multiplying `z = x + yi` by `a` is the real linear map

`[[p, -q], [q, p]] [x, y]^T`.

That matrix has two visible jobs. Its columns are perpendicular and have the same length, so it is a rotation followed by a dilation. The dilation factor is `|a|`; the rotation angle is the amplitude of `a`. The next artifact makes that statement inspectable by applying the same multiplier to a small frame, a triangle, and a chosen point. If the multiplier is correct, every length ratio should be scaled by the same factor and every angle should be shifted by the same amount.

In [ ]:
u = 1.15 + 0.55j
v = -0.35 + 1.05j
a = 1.25 + 0.75j
z = 0.72 + 0.64j
M = np.array([[a.real, -a.imag], [a.imag, a.real]])
z_vec = np.array([z.real, z.imag])
product_from_matrix = M @ z_vec
product_complex = a * z

fig, axs = plt.subplots(1, 2, figsize=(13, 5.4))
ax = axs[0]
for point, label, color in [(0j, "0", COLORS["ink"]), (u, "u", COLORS["blue"]), (v, "v", COLORS["green"]), (u + v, "u + v", COLORS["red"])]:
    ax.scatter([point.real], [point.imag], s=55, color=color)
    ax.text(point.real + 0.05, point.imag + 0.05, label, fontsize=10)
for start, end, color in [(0j, u, COLORS["blue"]), (0j, v, COLORS["green"]), (u, u + v, COLORS["green"]), (v, u + v, COLORS["blue"]), (0j, u + v, COLORS["red"])]:
    ax.add_patch(FancyArrowPatch((start.real, start.imag), (end.real, end.imag), arrowstyle="->", mutation_scale=12, lw=1.8, color=color))
ax.set_title("addition: parallelogram in the Argand plane")
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-0.7, 1.7)
ax.set_ylim(-0.2, 1.9)
ax.grid(alpha=0.25)
ax.set_xlabel("real")
ax.set_ylabel("imaginary")

ax = axs[1]
shape = np.array([0.1 + 0.05j, 0.95 + 0.15j, 0.45 + 0.82j, 0.1 + 0.05j])
image = a * shape
ax.plot(shape.real, shape.imag, marker="o", color=COLORS["blue"], lw=2.5, label="original")
ax.plot(image.real, image.imag, marker="o", color=COLORS["red"], lw=2.5, label="multiplied by a")
for point, label, color in [(z, "z", COLORS["green"]), (product_complex, "a z", COLORS["red"]), (a, "a", COLORS["purple"])]:
    ax.scatter([point.real], [point.imag], s=65, color=color)
    ax.text(point.real + 0.05, point.imag + 0.05, label, fontsize=10)
ax.add_patch(FancyArrowPatch((0, 0), (z.real, z.imag), arrowstyle="->", mutation_scale=12, lw=1.7, color=COLORS["green"]))
ax.add_patch(FancyArrowPatch((0, 0), (product_complex.real, product_complex.imag), arrowstyle="->", mutation_scale=12, lw=1.7, color=COLORS["red"]))
ax.set_title("multiplication: scale by |a| and rotate by arg(a)")
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-0.2, 1.8)
ax.set_ylim(-0.1, 2.0)
ax.grid(alpha=0.25)
ax.legend(loc="upper left")
ax.set_xlabel("real")
ax.set_ylabel("imaginary")

arith_path = save_png(fig, "argand-addition-multiplication.png")

matrix_residual = float(np.linalg.norm(product_from_matrix - np.array([product_complex.real, product_complex.imag])))
scale_residual = abs(abs(a * z) - abs(a) * abs(z))
argument_residual = angle_diff(cmath.phase(a * z), cmath.phase(a) + cmath.phase(z))
computed_checks["complex_arithmetic"] = {
    "matrix_product_residual": matrix_residual,
    "modulus_product_residual": float(scale_residual),
    "argument_addition_residual": argument_residual,
    "multiplier_modulus": abs(a),
    "multiplier_argument": cmath.phase(a),
}
assert matrix_residual < 1e-12
assert scale_residual < 1e-12
assert abs(argument_residual) < 1e-12

display_many([arith_path])


## 3. Modulus, Amplitude, And De Moivre

A complex number away from the origin has two polar coordinates: its modulus `r = |z|` and its amplitude, or argument, `theta`. Multiplication becomes simple in these coordinates: moduli multiply and amplitudes add. A unit complex number is therefore a pure rotation. Repeating it `n` times walks by equal angular steps.

That is the visual content of De Moivre's theorem. If `omega = cos(2*pi/n) + i sin(2*pi/n)`, then the powers `1, omega, omega^2, ..., omega^(n-1)` are the vertices of a regular `n`-gon, and the next power returns to `1`. If the modulus is not one, the same rule produces a spiral: the angle increments are still steady, but each step also changes radius by a fixed factor. The figure below pairs these two behaviors because the chapter explicitly uses both the unit-circle root story and the real-power/equiangular-spiral story.

In [ ]:
n = 7
omega = cmath.exp(2j * math.pi / n)
roots = np.array([omega**k for k in range(n)])
root_angles = np.unwrap(np.angle(roots))
angle_steps = np.diff(np.r_[root_angles, root_angles[0] + 2 * np.pi])

mu = 1.075
alpha = 0.42
powers = np.array([(mu * cmath.exp(1j * alpha)) ** t for t in np.linspace(0, 16, 360)])
integer_powers = np.array([(mu * cmath.exp(1j * alpha)) ** k for k in range(17)])

fig, axs = plt.subplots(1, 2, figsize=(13, 5.5))
ax = axs[0]
ax.add_patch(Circle((0, 0), 1, fill=False, ec=COLORS["muted"], lw=1.2, ls="--"))
ax.plot(np.r_[roots.real, roots[0].real], np.r_[roots.imag, roots[0].imag], color=COLORS["blue"], lw=2.2)
ax.scatter(roots.real, roots.imag, s=70, color=COLORS["orange"], zorder=3)
for k, root in enumerate(roots):
    ax.text(root.real * 1.12, root.imag * 1.12, f"omega^{k}", ha="center", va="center", fontsize=8)
ax.set_title("unit powers are regular polygon vertices")
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-1.35, 1.35)
ax.set_ylim(-1.35, 1.35)
ax.grid(alpha=0.25)
ax.set_xlabel("real")
ax.set_ylabel("imaginary")

ax = axs[1]
ax.plot(powers.real, powers.imag, color=COLORS["purple"], lw=2.2, label="real powers")
ax.scatter(integer_powers.real, integer_powers.imag, color=COLORS["red"], s=36, label="integer powers")
for k in [0, 4, 8, 12, 16]:
    p = integer_powers[k]
    ax.text(p.real + 0.04, p.imag + 0.04, str(k), fontsize=9)
ax.set_title("non-unit powers trace an equiangular spiral")
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=0.25)
ax.legend(loc="upper left")
ax.set_xlabel("real")
ax.set_ylabel("imaginary")

roots_path = save_png(fig, "modulus-amplitude-demoivre-roots.png")

root_rows = []
for k, root in enumerate(roots):
    root_rows.append({
        "k": k,
        "real": float(root.real),
        "imaginary": float(root.imag),
        "modulus": float(abs(root)),
        "power_residual": float(abs(root**n - 1)),
    })
roots_table = write_csv(TABLES / "roots-of-unity-checks.csv", root_rows)
table_artifacts.append(roots_table)
computed_checks["modulus_amplitude"] = {
    "n": n,
    "omega_power_residual": float(abs(omega**n - 1)),
    "angle_step_mean": float(angle_steps.mean()),
    "angle_step_std": float(angle_steps.std()),
    "spiral_radial_ratio": mu,
    "spiral_angle_step": alpha,
}
assert abs(omega**n - 1) < 1e-12
assert angle_steps.std() < 1e-12

display_many([roots_path])


## 4. Euler's Formula As A Converging Path

The source span presents Euler's formula after expanding the exponential, sine, and cosine series. The point of doing that computationally is not to reprove analysis from scratch. It is to make the identity visible: the partial sums of `exp(i theta)` move through the complex plane and converge to the point whose coordinates are `(cos theta, sin theta)`. The real terms gather into cosine; the imaginary terms gather into sine.

This is also where the famous special case `exp(i*pi) + 1 = 0` becomes geometrically small. The exponential has walked halfway around the unit circle, reached `-1`, and the extra `+1` returns it to the origin. The computation below records convergence errors rather than relying on the picture alone.

In [ ]:
theta = 1.35
terms = np.arange(1, 23)
partial_sums = []
for m in terms:
    s = sum((1j * theta) ** k / math.factorial(k) for k in range(m))
    partial_sums.append(s)
partial_sums = np.array(partial_sums)
target = math.cos(theta) + 1j * math.sin(theta)
errors = np.abs(partial_sums - target)
unit_circle = np.exp(1j * np.linspace(0, 2 * np.pi, 500))

theta_samples = np.linspace(-math.pi, math.pi, 181)
series_errors = []
for t in theta_samples:
    approx = sum((1j * t) ** k / math.factorial(k) for k in range(14))
    exact = math.cos(t) + 1j * math.sin(t)
    series_errors.append(abs(approx - exact))
series_errors = np.array(series_errors)

fig, axs = plt.subplots(1, 2, figsize=(13, 5.2))
ax = axs[0]
ax.plot(unit_circle.real, unit_circle.imag, color=COLORS["muted"], lw=1.2, ls="--", label="unit circle")
ax.plot(partial_sums.real, partial_sums.imag, marker="o", color=COLORS["blue"], lw=2.0, label="partial sums")
ax.scatter([target.real], [target.imag], color=COLORS["red"], s=80, label="cos(theta)+i sin(theta)")
for idx in [0, 1, 2, 4, 8, 13, 21]:
    ax.text(partial_sums[idx].real + 0.025, partial_sums[idx].imag + 0.025, str(terms[idx]), fontsize=8)
ax.set_aspect("equal", adjustable="box")
ax.set_title("partial sums of exp(i theta)")
ax.grid(alpha=0.25)
ax.legend(loc="lower left")
ax.set_xlabel("real")
ax.set_ylabel("imaginary")

ax = axs[1]
ax.semilogy(terms, errors, marker="o", color=COLORS["green"], lw=2.2, label="error at theta = 1.35")
ax.semilogy([14], [series_errors.max()], marker="s", color=COLORS["orange"], label="max error on [-pi, pi], 14 terms")
ax.set_title("series convergence makes the identity checkable")
ax.set_xlabel("number of exponential terms")
ax.set_ylabel("absolute error")
ax.grid(alpha=0.25, which="both")
ax.legend()

series_path = save_png(fig, "euler-formula-series-convergence.png")

euler_identity_residual = abs(cmath.exp(1j * math.pi) + 1)
computed_checks["euler_formula"] = {
    "theta": theta,
    "final_partial_sum_error": float(errors[-1]),
    "max_14_term_error_on_minus_pi_pi": float(series_errors.max()),
    "euler_identity_residual": float(euler_identity_residual),
    "sympy_exp_i_pi_plus_one": str(sp.simplify(sp.exp(sp.I * sp.pi) + 1)),
}
assert errors[-1] < 1e-14
assert euler_identity_residual < 1e-12

display_many([series_path])


## 5. Roots Of Equations As Plane Configurations

Over the complex numbers, many equations stop being blocked by sign or dimension. A quadratic with real coefficients may leave the real axis, but its roots still appear as mirror-image points. A root of unity problem distributes roots evenly around a circle. A general polynomial gives a finite constellation whose correctness is not visual symmetry but residual size: substitute each computed root back into the polynomial and the value should be close to zero.

The next cell does not try to reproduce the chapter's exercises. Instead it builds a reusable inspection habit. For each polynomial, it plots the roots, records their moduli and arguments, and writes a residual table. A learner can change the coefficients and ask whether the geometric pattern changes because of coefficient symmetry, degree, or accidental numerical choices.

In [ ]:
polynomials = {
    "z^5 - 1": np.array([1, 0, 0, 0, 0, -1], dtype=complex),
    "z^3 - (1 + 2i)": np.array([1, 0, 0, -(1 + 2j)], dtype=complex),
    "z^4 + z + 1": np.array([1, 0, 0, 1, 1], dtype=complex),
}

fig, axs = plt.subplots(1, 3, figsize=(15, 4.7))
root_rows = []
for ax, (name, coeffs) in zip(axs, polynomials.items()):
    roots_for_poly = np.roots(coeffs)
    residuals = np.polyval(coeffs, roots_for_poly)
    unit = np.exp(1j * np.linspace(0, 2 * np.pi, 400))
    ax.plot(unit.real, unit.imag, color=COLORS["muted"], lw=1, ls="--")
    ax.scatter(roots_for_poly.real, roots_for_poly.imag, s=72, color=COLORS["blue"])
    for j, root in enumerate(roots_for_poly):
        ax.text(root.real + 0.04, root.imag + 0.04, f"r{j+1}", fontsize=8)
        root_rows.append({
            "polynomial": name,
            "root_index": j + 1,
            "real": float(root.real),
            "imaginary": float(root.imag),
            "modulus": float(abs(root)),
            "argument": float(cmath.phase(root)),
            "residual_abs": float(abs(residuals[j])),
        })
    ax.axhline(0, color=COLORS["ink"], lw=0.8)
    ax.axvline(0, color=COLORS["ink"], lw=0.8)
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(name)
    ax.grid(alpha=0.25)
    ax.set_xlabel("real")
    ax.set_ylabel("imaginary")

fig.suptitle("Roots of equations are finite configurations in the complex plane", fontsize=14)
roots_constellation_path = save_png(fig, "polynomial-root-constellations.png")
root_table = write_csv(TABLES / "polynomial-root-residuals.csv", root_rows)
table_artifacts.append(root_table)
max_root_residual = max(row["residual_abs"] for row in root_rows)
computed_checks["roots_of_equations"] = {
    "polynomial_count": len(polynomials),
    "root_count": len(root_rows),
    "max_residual": float(max_root_residual),
}
assert max_root_residual < 1e-10

display_many([roots_constellation_path])


## 6. Conformal Transformations And Homographies

The final source sections move from complex arithmetic to transformations of the complex plane. Translations `z -> z + b` move every point by the same vector. Multiplications `z -> az` are dilative rotations about the origin, and `z -> a(z - c) + c` moves the center. Conjugation reflects across the real axis. Inversion `z -> 1/z` turns the unit circle inside out while keeping the circle fixed pointwise as a set. Combining these elementary operations leads to linear fractional transformations, often written `(az + b)/(cz + d)`, with `ad - bc` nonzero.

Two facts matter for a standalone geometry notebook. First, these maps send generalized circles, meaning ordinary circles and lines, to generalized circles. Second, wherever the derivative is nonzero and the map is holomorphic, small angles are preserved. The static figure shows grid and circle images under several maps. The Plotly lab lets the same sample curves be toggled under different transformations, because the best way to learn homographies is to watch familiar lines become circles and back again.

In [ ]:
def mobius(z: np.ndarray, a: complex, b: complex, c: complex, d: complex) -> np.ndarray:
    return (a * z + b) / (c * z + d)

def finite_points(points: np.ndarray, bound: float = 8.0) -> np.ndarray:
    mask = np.isfinite(points.real) & np.isfinite(points.imag) & (np.abs(points) < bound)
    return points[mask]

maps = {
    "translation z + b": lambda z: z + (0.55 + 0.35j),
    "dilative rotation a z": lambda z: (1.15 * cmath.exp(0.55j)) * z,
    "inversion 1/z": lambda z: 1 / z,
    "homography (az+b)/(cz+d)": lambda z: mobius(z, 1.0 + 0.35j, 0.35 - 0.15j, 0.25 + 0.18j, 1.0 - 0.1j),
}

t = np.linspace(-1.7, 1.7, 220)
grid_curves = []
for x0 in [-1.2, -0.6, 0.6, 1.2]:
    grid_curves.append(x0 + 1j * t)
for y0 in [-1.2, -0.6, 0.6, 1.2]:
    grid_curves.append(t + 1j * y0)
theta_grid = np.linspace(0, 2 * np.pi, 260)
grid_curves.append(0.95 * np.exp(1j * theta_grid))
grid_curves.append((0.45 + 0.25j) + 0.55 * np.exp(1j * theta_grid))

fig, axs = plt.subplots(2, 2, figsize=(12, 10))
for ax, (name, func) in zip(axs.flat, maps.items()):
    for curve in grid_curves:
        shifted = np.where(np.abs(curve) < 0.08, np.nan + 1j * np.nan, curve)
        mapped = finite_points(func(shifted))
        if len(mapped) > 2:
            ax.plot(mapped.real, mapped.imag, color=COLORS["blue"], lw=1.0, alpha=0.7)
    ax.axhline(0, color=COLORS["ink"], lw=0.7, alpha=0.55)
    ax.axvline(0, color=COLORS["ink"], lw=0.7, alpha=0.55)
    ax.set_title(name)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-3.1, 3.1)
    ax.set_ylim(-3.1, 3.1)
    ax.grid(alpha=0.2)
    ax.set_xlabel("real")
    ax.set_ylabel("imaginary")

fig.suptitle("Similarities, inversion, and homographies preserve generalized circle structure", fontsize=14)
conformal_static_path = save_png(fig, "circle-preserving-conformal-maps.png")

base_curves = {
    "unit circle": np.exp(1j * theta_grid),
    "offset circle": (0.55 + 0.25j) + 0.48 * np.exp(1j * theta_grid),
    "vertical line": 0.35 + 1j * np.linspace(-1.5, 1.5, 260),
    "horizontal line": np.linspace(-1.5, 1.5, 260) + 0.75j,
}
fig_lab = go.Figure()
trace_count_per_map = len(base_curves)
map_names = list(maps.keys())
for map_index, (name, func) in enumerate(maps.items()):
    for curve_name, curve in base_curves.items():
        safe_curve = np.where(np.abs(curve) < 0.05, np.nan + 1j * np.nan, curve)
        mapped = finite_points(func(safe_curve), bound=7.0)
        fig_lab.add_trace(go.Scatter(
            x=mapped.real,
            y=mapped.imag,
            mode="lines",
            name=curve_name,
            visible=(map_index == 0),
            hovertemplate=f"{name}<br>{curve_name}<br>x=%{{x:.3f}}<br>y=%{{y:.3f}}<extra></extra>",
        ))
buttons = []
for map_index, name in enumerate(map_names):
    visible = [False] * (trace_count_per_map * len(map_names))
    start = map_index * trace_count_per_map
    for j in range(trace_count_per_map):
        visible[start + j] = True
    buttons.append({"label": name, "method": "update", "args": [{"visible": visible}, {"title": f"Complex map lab: {name}"}]})
fig_lab.update_layout(
    title="Complex map lab: translation z + b",
    xaxis_title="real",
    yaxis_title="imaginary",
    width=860,
    height=640,
    template="plotly_white",
    updatemenus=[{"buttons": buttons, "direction": "down", "x": 0.02, "y": 1.14}],
    legend_title_text="sample curve",
)
fig_lab.update_yaxes(scaleanchor="x", scaleratio=1)
html_path = HTML / "linear-fractional-map-lab.html"
if html_path.exists():
    html_path.unlink()
fig_lab.write_html(html_path, include_plotlyjs="cdn", full_html=True)
visual_artifacts.append(html_path)

z0 = 0.35 + 0.42j
eps = 1e-5
v1 = eps * (1 + 0j)
v2 = eps * cmath.exp(1.1j)
F = maps["homography (az+b)/(cz+d)"]
def local_vector(v: complex) -> complex:
    return F(np.array([z0 + v]))[0] - F(np.array([z0]))[0]
angle_before = cmath.phase(v2 / v1)
angle_after = cmath.phase(local_vector(v2) / local_vector(v1))
angle_preservation_error = abs(angle_diff(angle_after, angle_before))

a_sym, b_sym, c_sym, d_sym = 1 + sp.I / 3, sp.Rational(1, 3) - sp.I / 7, sp.Rational(1, 4) + sp.I / 5, 1 - sp.I / 10
det_value = sp.simplify(a_sym * d_sym - b_sym * c_sym)
computed_checks["conformal_transformations"] = {
    "local_angle_before": float(angle_before),
    "local_angle_after": float(angle_after),
    "angle_preservation_error": float(angle_preservation_error),
    "sample_homography_determinant": str(det_value),
    "plotly_maps": map_names,
}
assert angle_preservation_error < 1e-5
assert det_value != 0

display_many([conformal_static_path, html_path])


## Reading The Chapter As A Proof Environment

The proof pattern across the chapter is: translate an algebraic expression into a geometric operation, observe the invariant, and then compute the invariant in a form that can fail if the claim is false. Complex multiplication is not merely a formula because the matrix check confirms the same operation in real coordinates. De Moivre's theorem is not merely a mnemonic because powers of a unit complex number close exactly after `n` steps. Euler's formula is not merely famous notation because the exponential series converges to the trigonometric point. Polynomial roots are not merely a list because residuals certify that the plotted points really solve the equations. Homographies are not merely elegant symbols because their local angle behavior and circle/line images can be tested.

There is also a useful warning. Many of the figures look symmetric because the examples were chosen to reveal structure, but the code uses asymmetric probes wherever possible. Symmetry can seduce the eye into accepting a false general statement. A residual, determinant, angle check, or table of values forces the picture to answer a sharper question.

In [ ]:
manifest_rows = []
for artifact in visual_artifacts:
    manifest_rows.append({
        "artifact": rel(artifact),
        "role": "visual",
        "concept": artifact.stem.replace("-", " "),
    })
for artifact in table_artifacts:
    manifest_rows.append({
        "artifact": rel(artifact),
        "role": "table",
        "concept": artifact.stem.replace("-", " "),
    })

manifest_path = write_csv(TABLES / "artifact-manifest.csv", manifest_rows)
table_artifacts.append(manifest_path)

summary_payload = {
    "chapter": CHAPTER_NO,
    "title": CHAPTER["title"],
    "source_printed_pages": CHAPTER["printed"],
    "source_pdf_pages": CHAPTER["pdf"],
    "visuals": [rel(path) for path in visual_artifacts],
    "tables": [rel(path) for path in table_artifacts],
    "checks": computed_checks,
}
summary_path = write_json(CHECKS / "visual-summary.json", summary_payload)
check_artifacts.append(summary_path)
legacy_summary_path = write_json(CHECKS / "visual_summary.json", summary_payload)
check_artifacts.append(legacy_summary_path)

final_sanity = {
    "visual_count": len(visual_artifacts),
    "table_count": len(table_artifacts),
    "max_polynomial_residual": computed_checks["roots_of_equations"]["max_residual"],
    "homography_angle_error": computed_checks["conformal_transformations"]["angle_preservation_error"],
    "euler_identity_residual": computed_checks["euler_formula"]["euler_identity_residual"],
}
final_sanity_path = write_json(CHECKS / "final-sanity-summary.json", final_sanity)
check_artifacts.append(final_sanity_path)

assert_artifacts(visual_artifacts + table_artifacts + check_artifacts, min_bytes=80)
assert final_sanity["visual_count"] >= 6
assert final_sanity["max_polynomial_residual"] < 1e-10
assert final_sanity["homography_angle_error"] < 1e-5
assert final_sanity["euler_identity_residual"] < 1e-12
assert not (FIG / "concept_configuration.svg").exists()
assert not (FIG / "parameter_experiment.svg").exists()
assert not (TABLES / "artifact_manifest.csv").exists()

summary_payload


## Takeaways

- The complex plane is a geometric extension of the number system: `x + iy` is both a number and a point with coordinates `(x, y)`.
- Addition is vector translation; multiplication by `a + bi` is a rotation through `arg(a)` followed by scaling by `|a|`.
- Modulus and amplitude are the coordinates in which products, powers, reciprocals, and roots become simple geometric actions.
- De Moivre's theorem turns powers of unit complex numbers into regular polygons and turns non-unit powers into spiral behavior.
- Euler's formula links the exponential series to circular motion; the special case `exp(i*pi) + 1 = 0` is a half-turn plus a return to the origin.
- Roots of equations are plane configurations that should be plotted and checked by residuals, not only written as algebraic answers.
- Linear fractional transformations organize similarities, inversions, and circle-preserving maps; local angle checks make the word conformal computationally inspectable.